In [22]:
%pip install langchain
%pip install langchain_community chromadb
%pip install langchain_huggingface



In [23]:
from google.colab import userdata
my_token = userdata.get('HF_MODEL')

In [24]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm_endpoint = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    provider="featherless-ai",   # explicitly pick the provider that hosts this model
    task="conversational",
    huggingfacehub_api_token=my_token
)

chat_model = ChatHuggingFace(llm=llm_endpoint)


In [25]:
print(chat_model.invoke("What is the capital of India?").content)

 The capital city of India is New Delhi. It is important to note that New Delhi is not a state, but a union territory and the administrative headquarters of the Government of India. It is located in the northern part of India and was officially declared as the capital of India on February 12, 1931.


In [26]:
from langchain_huggingface import HuggingFaceEmbeddings


In [27]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # small, fast, good default
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [39]:
vector_store.delete_collection()

In [40]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [41]:
print(type(doc1))
print(doc1.page_content)
print(doc1.metadata)

<class 'langchain_core.documents.base.Document'>
Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
{'team': 'Royal Challengers Bangalore'}


In [42]:

from langchain_community.vectorstores import Chroma

In [43]:
vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [44]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [45]:
vector_store.add_documents(docs)

['ad59d09e-161e-407c-8ee5-d19b86fcaa2e',
 'bfebe665-601f-4e7d-8c09-8b98871a2b2f',
 '809c769e-193f-4fdf-b795-7a898af4b25b',
 '66a6f58d-56e0-4cdb-adcf-41e542177a0a',
 'b35c6f9d-aaab-4550-8c46-75cdbb81209d']

In [46]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['ad59d09e-161e-407c-8ee5-d19b86fcaa2e',
  'bfebe665-601f-4e7d-8c09-8b98871a2b2f',
  '809c769e-193f-4fdf-b795-7a898af4b25b',
  '66a6f58d-56e0-4cdb-adcf-41e542177a0a',
  'b35c6f9d-aaab-4550-8c46-75cdbb81209d'],
 'embeddings': array([[ 0.00994725,  0.06914335, -0.0514712 , ..., -0.0354334 ,
          0.01284813,  0.01248285],
        [ 0.00127746,  0.0312985 , -0.02375378, ..., -0.00518364,
         -0.03280616,  0.02737711],
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        [ 0.01873975,  0.04382844, -0.04304259, ..., -0.07801618,
         -0.07840681, -0.00304193]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [47]:
ans = vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

for i in range(len(ans)):
  print(ans[i].page_content)

Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.
Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.


In [48]:
print(vector_store._collection.count())  # total number of vectors stored

5


In [49]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436007499694824),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8909369707107544)]

In [50]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='ad59d09e-161e-407c-8ee5-d19b86fcaa2e', document=updated_doc1)


In [51]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['ad59d09e-161e-407c-8ee5-d19b86fcaa2e',
  'bfebe665-601f-4e7d-8c09-8b98871a2b2f',
  '809c769e-193f-4fdf-b795-7a898af4b25b',
  '66a6f58d-56e0-4cdb-adcf-41e542177a0a',
  'b35c6f9d-aaab-4550-8c46-75cdbb81209d'],
 'embeddings': array([[-0.00233746,  0.05902081, -0.04774044, ..., -0.07264049,
          0.00276782, -0.00344088],
        [ 0.00127746,  0.0312985 , -0.02375378, ..., -0.00518364,
         -0.03280616,  0.02737711],
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        [ 0.01873975,  0.04382844, -0.04304259, ..., -0.07801618,
         -0.07840681, -0.00304193]]),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a sin

In [52]:
# delete document
vector_store.delete(ids=['ad59d09e-161e-407c-8ee5-d19b86fcaa2e'])

In [53]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['bfebe665-601f-4e7d-8c09-8b98871a2b2f',
  '809c769e-193f-4fdf-b795-7a898af4b25b',
  '66a6f58d-56e0-4cdb-adcf-41e542177a0a',
  'b35c6f9d-aaab-4550-8c46-75cdbb81209d'],
 'embeddings': array([[ 0.00127746,  0.0312985 , -0.02375378, ..., -0.00518364,
         -0.03280616,  0.02737711],
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        [ 0.01873975,  0.04382844, -0.04304259, ..., -0.07801618,
         -0.07840681, -0.00304193]]),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one 

In [54]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 99.1 MB/s eta 0:00:00


In [60]:
documents2 = [
    Document(page_content="LangChain is a framework for building applications with LLMs.", metadata={"source": "docs"}),
    Document(page_content="FAISS is a library for efficient similarity search of dense vectors.", metadata={"source": "meta"}),
    Document(page_content="RAG combines retrieval systems with generative models.", metadata={"source": "ai"})
]

In [61]:
from langchain_community.vectorstores import FAISS

vector_store2 = FAISS.from_documents(
    documents=documents2,
    embedding=embedding_model,
)

In [63]:
for doc_id, doc in vector_store2.docstore._dict.items():
    print(f"Document ID: {doc_id}")
    print(f"Page Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print("\n---\n")

Document ID: 40057da7-977a-43f1-a6b4-6d29a4582132
Page Content: LangChain is a framework for building applications with LLMs.
Metadata: {'source': 'docs'}

---

Document ID: c34a7994-2196-490f-a755-f03744ba68e4
Page Content: FAISS is a library for efficient similarity search of dense vectors.
Metadata: {'source': 'meta'}

---

Document ID: 2f2a6057-c8d9-4db6-97de-f006ae220de6
Page Content: RAG combines retrieval systems with generative models.
Metadata: {'source': 'ai'}

---



In [64]:
vector_store2.add_documents(docs)

['4e2e91ab-ef11-4327-9d5f-51372b2d6d99',
 '656a60ef-8d2c-45c2-8279-4060dba0abbc',
 '36bb7646-5523-4469-9c7e-650f8f1a7b90',
 '47baad57-54a9-4fe0-bd26-f4084cf5f549',
 '76b28ae7-11c3-466f-b2f7-8fededce68b4']

In [65]:
for doc_id, doc in vector_store2.docstore._dict.items():
    print(f"Document ID: {doc_id}")
    print(f"Page Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print("\n---\n")


Document ID: 40057da7-977a-43f1-a6b4-6d29a4582132
Page Content: LangChain is a framework for building applications with LLMs.
Metadata: {'source': 'docs'}

---

Document ID: c34a7994-2196-490f-a755-f03744ba68e4
Page Content: FAISS is a library for efficient similarity search of dense vectors.
Metadata: {'source': 'meta'}

---

Document ID: 2f2a6057-c8d9-4db6-97de-f006ae220de6
Page Content: RAG combines retrieval systems with generative models.
Metadata: {'source': 'ai'}

---

Document ID: 4e2e91ab-ef11-4327-9d5f-51372b2d6d99
Page Content: Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.
Metadata: {'team': 'Royal Challengers Bangalore'}

---

Document ID: 656a60ef-8d2c-45c2-8279-4060dba0abbc
Page Content: Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ab

In [66]:
# search documents
vector_store2.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='47baad57-54a9-4fe0-bd26-f4084cf5f549', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='656a60ef-8d2c-45c2-8279-4060dba0abbc', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [67]:
# search with similarity score
vector_store2.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='47baad57-54a9-4fe0-bd26-f4084cf5f549', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  np.float32(0.9693601)),
 (Document(id='656a60ef-8d2c-45c2-8279-4060dba0abbc', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  np.float32(1.1493449))]